# Compact models vs much larger models: two head-to-heads

**A. ASO knockdown:** our compact LightGBM (tens of thousands of numbers) vs the released **OligoAI** (RiNALMo-giga fine-tune, ~650M parameters).
**B. mtDNA variant effect:** our conservation + annotation scorer (~15 parameters + a 0.5 MB count table) vs the **Nucleotide Transformer 500M**.

Use a **GPU runtime** (Runtime -> Change runtime type -> A100 or L4; T4 also works for everything here because no FlashAttention is needed).
Both comparisons were designed before the locked data were scored. Plumbing was tested with shuffled labels and a random stand-in model; no real result has been seen.

**Licences / cautions.** OligoAI *code* is Apache-2.0; its *weights* have no stated licence (use for private benchmarking, do not redistribute predictions).
Nucleotide Transformer v1 weights are CC-BY-NC-SA-4.0 (non-commercial). The ASO table is CC-BY-4.0. The OligoAI checkpoint is a pickle: it is read only with `weights_only=True` and one allowed class.

In [ ]:
!nvidia-smi -L
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

## 0. Get the files
Upload `hct_compare.zip` (from the chat) to Colab, or put it in Drive and adjust the path.

In [ ]:
from google.colab import files
up = files.upload()              # choose hct_compare.zip
!unzip -q -o hct_compare.zip -d /content
%cd /content/hct_compare
!ls

In [ ]:
!pip -q install ViennaRNA lightgbm biopython "transformers>=4.40" huggingface_hub scikit-learn scipy pandas

## A. ASO: compact model vs OligoAI

In [ ]:
import os, hashlib, urllib.request
os.makedirs('data', exist_ok=True)
url = 'https://zenodo.org/api/records/20794660/files/aso_inhibitions_21_08_25_incl_context_w_flank_50_df.csv.gz/content'
dest = 'data/aso.csv.gz'
if not os.path.exists(dest): urllib.request.urlretrieve(url, dest)
md5 = hashlib.md5(open(dest,'rb').read()).hexdigest(); print(md5, md5 == '0efae5ae62d8d2c222c9caf5e01ced07')

In [ ]:
!python compare/aso_compare.py prepare --csv data/aso.csv.gz --out work

In [ ]:
from huggingface_hub import hf_hub_download
ckpt = hf_hub_download('barneyhill/OligoAI', 'OligoAI_11_09_25.ckpt')      # ~3 GB; licence not stated on the model card
print(ckpt)

In [ ]:
# plumbing check on a random tiny checkpoint (seconds), then the real scoring (~10-30 min on a GPU)
!python compare/oligoai_score.py --selftest
!python compare/oligoai_score.py --csv data/aso.csv.gz --ckpt {ckpt} --ids work/ids_to_score.csv --out work/oligoai_pred.csv --check --bf16 --batch 64

In [ ]:
!python compare/aso_compare.py evaluate --csv data/aso.csv.gz --pred work/oligoai_pred.csv --cache cache --out work --boot 300

## B. mtDNA: compact scorer vs Nucleotide Transformer 500M

In [ ]:
!python mitolm/mito_lm.py download --workdir mito_work
!python mitolm/mito_lm.py corpus --workdir mito_work
!python mitolm/mito_lm.py export-variants --workdir mito_work --constraint-n 6000

In [ ]:
!python compare/mito_nt_compare.py --workdir mito_work --out mito_work/ext_nt500m.csv --name nt500m --fp16 --batch 64

In [ ]:
# 'small' has no checkpoint here, so only conservation / annotation / n-gram / Nucleotide Transformer are compared (upload mito_work/ckpt/small.pt first to add our own LM)
!python mitolm/mito_lm.py evaluate --workdir mito_work --sizes small --external mito_work/ext_nt500m.csv --constraint-n 6000

## Save everything

In [ ]:
!mkdir -p results && cp work/aso_compare_report.txt results/ && cp mito_work/results/report.txt results/mito_report.txt && cp mito_work/results/report_constraint.txt results/mito_report_constraint.txt
!zip -q -r results.zip results
files.download('results.zip')      # send me the three .txt files